# **Análisis Crítico de los Modelos - NPL**

In [3]:
import os
import json
import pickle
import pandas as pd

def cargar_metricas():
    resultados = []
    
    # 1. CARGAR XGBOOST (.csv)
    # Ruta corregida según tu árbol: '../Models/xgboost_results/metrics_xgboost.csv'
    xgb_path = '../Models/xgboost_results/metrics_xgboost.csv'
    if os.path.exists(xgb_path):
        df_xgb = pd.read_csv(xgb_path)
        resultados.append({
            'Modelo': 'TF-IDF + XGBoost',
            'Accuracy': df_xgb['Accuracy'].values[0],
            'Precision': df_xgb['Precision'].values[0],
            'Recall': df_xgb['Recall'].values[0],
            'F1': df_xgb['F1'].values[0],
            'ROC-AUC': df_xgb['ROC-AUC'].values[0] if 'ROC-AUC' in df_xgb.columns else df_xgb['AUC'].values[0]
        })

    # 2. CARGAR CNN-1D (.pkl)
    # Agregamos múltiples opciones de llaves para evitar el NaN en ROC-AUC
    cnn_path = '../Models/cnn_results/metrics.pkl'
    if os.path.exists(cnn_path):
        with open(cnn_path, 'rb') as f:
            metrics_cnn = pickle.load(f)
        resultados.append({
            'Modelo': 'CNN-1D',
            'Accuracy': metrics_cnn.get('Accuracy', metrics_cnn.get('accuracy')),
            'Precision': metrics_cnn.get('Precision', metrics_cnn.get('precision')),
            'Recall': metrics_cnn.get('Recall', metrics_cnn.get('recall')),
            'F1': metrics_cnn.get('F1-score', metrics_cnn.get('F1', metrics_cnn.get('f1'))),
            'ROC-AUC': metrics_cnn.get('ROC-AUC', metrics_cnn.get('roc_auc', metrics_cnn.get('ROC_AUC', metrics_cnn.get('auc'))))
        })

    # 3. CARGAR FASTTEXT + BILSTM (.csv)
    fasttext_path = '../Models/FastText_BiLSTM_results/fasttext_bilstm_metrics.csv'
    if os.path.exists(fasttext_path):
        df_ft = pd.read_csv(fasttext_path)
        resultados.append({
            'Modelo': 'FastText + BiLSTM',
            'Accuracy': df_ft['Accuracy'].values[0],
            'Precision': df_ft['Precision'].values[0],
            'Recall': df_ft['Recall'].values[0],
            'F1': df_ft['F1'].values[0],
            'ROC-AUC': df_ft['ROC-AUC'].values[0]
        })

    # 4. CARGAR WORD2VEC + BILSTM (.csv)
    # CORRECCIÓN DE RUTA: Carpeta termina en '_result' según tu estructura
    w2v_path = '../Models/Word2Vec_BiLSTM_result/bilstm_metrics.csv'
    if os.path.exists(w2v_path):
        df_w2v = pd.read_csv(w2v_path)
        resultados.append({
            'Modelo': 'Word2Vec + BiLSTM',
            'Accuracy': df_w2v['Accuracy'].values[0],
            'Precision': df_w2v['Precision'].values[0],
            'Recall': df_w2v['Recall'].values[0],
            'F1': df_w2v['F1'].values[0],
            'ROC-AUC': df_w2v['ROC-AUC'].values[0] if 'ROC-AUC' in df_w2v.columns else df_w2v['auc'].values[0]
        })

    # 5. CARGAR DISTILBERT (.json)
    # CORRECCIÓN DE RUTA: Carpeta en minúsculas y singular 'distilbert_result'
    bert_path = '../Models/distilbert_result/metrics.json'
    if os.path.exists(bert_path):
        with open(bert_path, 'r') as f:
            metrics_bert = json.load(f)
        resultados.append({
            'Modelo': 'DistilBERT (Transformer)',
            'Accuracy': metrics_bert.get('accuracy', metrics_bert.get('Accuracy')),
            'Precision': metrics_bert.get('precision', metrics_bert.get('Precision')),
            'Recall': metrics_bert.get('recall', metrics_bert.get('Recall')),
            'F1': metrics_bert.get('f1', metrics_bert.get('f1_score', metrics_bert.get('F1'))),
            'ROC-AUC': metrics_bert.get('roc_auc', metrics_bert.get('ROC-AUC', metrics_bert.get('auc')))
        })

    df_comparativo = pd.DataFrame(resultados)
    columnas_numericas = ['Accuracy', 'Precision', 'Recall', 'F1', 'ROC-AUC']
    df_comparativo[columnas_numericas] = df_comparativo[columnas_numericas].astype(float).round(4)
    
    return df_comparativo

if __name__ == "__main__":
    tabla_final = cargar_metricas()
    # Guardar a CSV
    tabla_final.to_csv('tabla_comparativa_modelos.csv', index=False)

In [4]:
tabla_final

,Modelo,Accuracy,Precision,Recall,F1,ROC-AUC
0,TF-IDF + XGBoost,0.4933,0.5781,0.4933,0.4441,0.8943
1,CNN-1D,0.5131,0.4788,0.5131,0.4772,0.8933
2,FastText + BiLSTM,0.6761,0.6577,0.6761,0.6569,0.9385
3,Word2Vec + BiLSTM,0.6901,0.6691,0.6901,0.6732,0.9351
4,DistilBERT (Transformer),0.8231,0.8251,0.8231,0.8195,0.9695


## **¿Transformers Vs Modelos Clásicos?**

Los resultados muestran una diferencia clara entre los modelos clásicos/tradicionales y los modelos basados en Transformers para la tarea de clasificación de texto. Los enfoques clásicos y secuenciales, como TF-IDF + XGBoost, CNN-1D y BiLSTM con embeddings preentrenados, alcanzan desempeños moderados, con accuracies entre 49% y 69% y valores de F1-score relativamente limitados. Aunque modelos como FastText + BiLSTM y Word2Vec + BiLSTM mejoran considerablemente respecto a TF-IDF y CNN-1D, todavía presentan dificultades para capturar completamente el contexto semántico y las relaciones complejas del lenguaje natural.

Por otro lado, el modelo Transformer basado en DistilBERT obtiene el mejor desempeño global en todas las métricas evaluadas, alcanzando una accuracy de 0.8231, un F1-score de 0.8195 y el mayor ROC-AUC (0.9695). Esto indica una capacidad superior para discriminar y clasificar correctamente las distintas categorías del problema multiclase.

La diferencia de rendimiento puede explicarse porque los Transformers utilizan mecanismos de atención (*self-attention*) que permiten comprender relaciones contextuales entre palabras a lo largo de toda la secuencia, mientras que los modelos clásicos dependen más de representaciones estáticas o de patrones locales/secuenciales limitados. Además, DistilBERT aprovecha conocimiento lingüístico previamente aprendido durante el preentrenamiento en grandes corpus de texto, lo que mejora significativamente la representación semántica y la generalización del modelo incluso con conjuntos de datos relativamente pequeños.

En conclusión, los resultados evidencian que los modelos Transformer superan claramente a los modelos clásicos y recurrentes en tareas modernas de procesamiento de lenguaje natural, ofreciendo representaciones más robustas, mejor comprensión contextual y un desempeño más estable en clasificación multiclase.


## **¿Impacto del Preprocesamiento?**

El preprocesamiento tuvo un impacto importante en el desempeño de los modelos, ya que permitió transformar los textos en representaciones numéricas adecuadas para el aprendizaje automático y reducir ruido presente en los datos. Técnicas como tokenización, padding, codificación de etiquetas y el uso de embeddings preentrenados (Word2Vec y FastText) ayudaron a mejorar la representación semántica de las palabras y facilitaron el entrenamiento de los modelos profundos.

Los resultados sugieren que las representaciones semánticas avanzadas tuvieron un efecto positivo sobre el rendimiento. Modelos basados en embeddings distribuidos y contexto lingüístico, como BiLSTM y especialmente DistilBERT, lograron métricas considerablemente superiores frente a enfoques más tradicionales como TF-IDF + XGBoost. Esto indica que preservar información contextual y relaciones semánticas entre palabras es fundamental en tareas de clasificación de texto.

Además, el uso de padding con una longitud fija permitió estandarizar las secuencias de entrada para las redes neuronales, mientras que la codificación one-hot de las etiquetas facilitó el aprendizaje multiclase. Sin embargo, las advertencias observadas en algunas métricas muestran que aún existen clases difíciles de representar adecuadamente, posiblemente debido a desbalance de clases o poca cantidad de ejemplos en ciertas categorías.

En el caso de DistilBERT, el impacto del preprocesamiento fue menor en comparación con los modelos clásicos, ya que los Transformers incorporan tokenización contextual y representaciones semánticas profundas preentrenadas. Esto demuestra que los modelos modernos dependen menos de ingeniería manual de características y más de representaciones aprendidas automáticamente durante el preentrenamiento.


## **¿Errores Comunes?**

Los errores más comunes observados durante el desarrollo y evaluación de los modelos están directamente relacionados con el desbalance de clases, la confusión semántica entre categorías similares y las limitaciones de generalización de ciertas arquitecturas.

El principal problema identificado fue el desbalance del dataset. Varias categorías poseen muy pocos ejemplos (*support* bajo), como **AUTOMOBILE** o **BPO**, provocando que algunos modelos no generaran ninguna predicción para estas clases. Esto produjo advertencias como `UndefinedMetricWarning` y métricas de precisión, recall y F1-score iguales a 0.00 en determinadas categorías. Como consecuencia, aunque algunos modelos presentaban valores altos de ROC-AUC, el rendimiento real en clasificación dura se deterioraba significativamente.

Otro error frecuente fue la confusión entre clases semánticamente cercanas. Las matrices de confusión mostraron solapamientos importantes entre categorías relacionadas profesionalmente, donde el modelo tendía a asignar textos a clases más frecuentes o con patrones léxicos similares. Esto evidencia que ciertos modelos tienen dificultades para aprender fronteras de decisión precisas en escenarios multiclase complejos.

También se observó una diferencia importante entre desempeño probabilístico y desempeño final de clasificación. Modelos como CNN-1D, TF-IDF + XGBoost y BiLSTM alcanzaron valores elevados de ROC-AUC, indicando buena capacidad de separación probabilística, pero sus métricas finales de Accuracy y F1-score fueron considerablemente menores. Esto demuestra que el modelo puede ordenar correctamente las probabilidades internas, pero falla al convertir esas probabilidades en etiquetas definitivas debido al umbral de decisión y al desbalance de datos.

Adicionalmente, algunos modelos mostraron problemas de sobreespecialización en ciertas categorías dominantes. En particular, arquitecturas como CNN-1D tendieron a generar sesgos hacia clases más representadas, reflejados visualmente en columnas dominantes dentro de las matrices de confusión. Esto limitó la capacidad de generalización hacia categorías minoritarias.

Finalmente, se identificó que las arquitecturas más simples o basadas únicamente en patrones locales, como TF-IDF + XGBoost y CNN-1D, presentan mayores dificultades para capturar el contexto semántico profundo del lenguaje natural. En contraste, modelos con embeddings contextuales y mecanismos de atención, como DistilBERT, lograron reducir significativamente estos errores y alcanzar una clasificación más estable y robusta entre las distintas categorías.